# 第56课 · 让梯度逆流而上——手推反向传播（backpropagation）：拓扑排序 × 链式法则（chain rule）

**目标**：实现 `Value.backward()`——拓扑排序 × 链式法则，让梯度逆流。

> **最关键 DL 课之一**：先用 3 个节点手推拓扑序与链式乘，再写代码。回链 L24。

🔗 **Aurora 连接**：这正是 PyTorch 的 `.backward()` 在做的事情；Month 2 训练循环（权重更新）依赖此机制。

← **上一课**　[L55 · Value 算子补全](L55_forward_pass.ipynb)

> 上节课学习了 **Value 算子补全**：pow、relu、tanh、exp 节点实现，计算图完整展开。  
> 本课将探讨 **反向传播手推**。

> **难度桥（L55 前向 → L56 反向）**：上一课你让数据"顺流而下"，从叶子算到输出 `L`（前向）；这一课把水闸反向打开，让梯度"逆流而上"，从 `L` 回到每个参数。算子、计算图、局部导数公式**全都没变**——唯一变的是**遍历方向**。所以别紧张：新东西其实只有"按什么顺序倒着走"这一件事。

## 本课剧情：顺序错了，梯度就错了

前两节课给每个算子装上了"局部反向函数"。但有个问题没解决：**按什么顺序调用这些函数？**

错误示范：随机顺序调用 `_backward()`。考虑计算图 `L = a*b + c`：
- 若先调 `a*b` 的 `_backward()`，此时 `(a*b).grad` 还是 0（`+` 还没传过来）→ `a.grad` 和 `b.grad` 算出来全是 0

**结论：必须从输出节点往输入节点逆向，而且每个节点得等它后面的节点都处理完，才轮到它自己。** 这就是 DAG 的**逆拓扑序（reverse topological order）**。

算法只需三步：

```
1. 从 L 出发，DFS 后序遍历 DAG → 得到拓扑序列 topo
2. L.grad = 1.0                  （dL/dL = 1，链式法则的起点）
3. for node in reversed(topo):   → 逆序调用每个节点的 _backward()
       node._backward()
```

**梯度累积陷阱**：同一节点 `a` 被两个算子使用时（`L = a*a`），梯度从两条路径传来，必须 `+=` 累积，不能用 `=` 覆盖。

本节任务：实现 `Value.backward()`，让 `L.backward()` 一键完成整图反向传播。

## 🤔 为什么工程师要发明它？(Why did engineers invent this?)

- **不用它会怎样？** 一个百万参数的网络，你得为每个参数手写一条偏导公式再逐点求值——写不完，也调不动。
- **它解决了什么真实问题？** 反向传播把"求所有参数的梯度"变成**一次图遍历**：复用前向已算好的中间值，一趟 `for` 循环就能把 `dL/d参数` 全部填好，代价和一次前向差不多。
- **后面哪里还会再用到？** L57 MLP 整网反向、L58 训练循环的 `w -= lr*w.grad`、L60 PyTorch 的 `.backward()`——全都是这套机制的放大版。

## ⚠️ 常见误解 (Common Pitfall)

> 不要把 Backprop 理解成某种"神秘的学习算法"。它其实**只是链式法则**——把 `dL/dx = dL/dg · dg/dx` 沿计算图按**逆拓扑序机械地展开**一遍，没有任何魔法，也不含"学习"。真正让网络学习的，是后面用这些梯度做的梯度下降（L58）。

In [1]:
import numpy as np

# ── Value 基础实现（含各运算的 _backward，backward 留待下方实现）──
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _bwd():
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _bwd
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _bwd():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _bwd
        return out

    def __neg__(self):        return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other

    def relu(self):
        out = Value(max(0, self.data), (self,), 'ReLU')
        def _bwd():
            self.grad += (out.data > 0) * out.grad
        out._backward = _bwd
        return out

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

print("Value 类加载完毕")

Value 类加载完毕


## 💧 直觉：把梯度想成水流（Water Pipe Analogy）

正式推导前，先建立一幅画面。把计算图想成一套**水管网络**，梯度就是里面流动的水：

- 输出节点 `L` 是**总水源**，往回灌入的水量固定为 1（这就是稍后的 `L.grad = 1.0`）。
- 每个节点收到**上游流量**（`out.grad`），乘上自己这段管子的**本地导数**（局部偏导），得到分给每个输入的流量，再往**下游**（输入方向）传。
  - 例：`out = a * b` 时，`a` 分到的流量 = 上游流量 × `b`（`a` 这段管子的本地导数正是 `b`）。
- 一个节点若有**两根管子汇入**（被复用，如 `L = a*a`），两股水要**相加**——这就是为什么用 `+=` 累积，而不是覆盖。
- **必须等一个节点所有上游的水都到齐，才能往下分**——这正是下面要讲的"逆拓扑序"。

记住这一句："**上游流量 × 本地导数，再往下游传**"，就是反向传播的全部。下面把这幅画面写成严格的算法。

### 记号澄清：`d` vs `∂`（全导数 vs 偏导数）

在推导反向传播时，你会看到两种记号：`dL/dx` 和 `∂L/∂x`。它们有什么区别？

- **`d` 表示全导数**（total derivative）：用于描述一个变量只通过**一条路径**影响另一个变量时的导数
- **`∂` 表示偏导数**（partial derivative）：用于描述多变量函数中，固定其他变量，对一个变量求导

**实例**：计算图 `L = a*b + c`
- `a` 只通过一条路径影响 `L`（`a → a*b → L`），所以写 `dL/da` 是没有歧义的
- 如果一个复杂函数 `f(a, b, c) = a*b + c`，求 `a` 的导数时通常写 `∂f/∂a`（表示固定 `b, c`）

**在反向传播中的实践**：
- 我们用 `dL/dx` 或 `∂L/∂x` 来表示"损失 `L` 对参数 `x` 的偏导"
- 这两个记号在反向传播的语境下通常可以互换，因为每个参数对输出的所有路径都被计算图自动跟踪了
- 数学上更严谨的记法是 `∂L/∂x`，但在深度学习代码中常见 `dL/dx` 的简写

**关键是理解含义**：`dL/dx` 表示的就是"如果 `x` 增加 1 单位，`L` 会变化多少"，无论用什么符号。

## 概念 1：拓扑排序保证梯度累积（gradient accumulation）顺序

计算图是一个 DAG（有向无环图）。节点 `v` 的 `_backward()` 要把 `v.grad` 继续分发给它的输入，而 `v.grad` 本身来自所有"用到了 `v.data` 的节点"传回的上游梯度——这些上游梯度必须先全部累积到位。

拓扑排序给出一个线性顺序，使得：若 `u → v`（`u` 依赖 `v`，即 `v` 是 `u` 的输入），则 `u` 排在 `v` 后面。
反向遍历此顺序，就能保证每个节点 `_backward()` 被调用时，它的 `.grad` 已经收齐了所有来自输出方向的上游梯度。

DFS 后序遍历（post-order）天然生成拓扑序：先递归访问所有子节点，访问完后再把当前节点加入列表。

```
topo = []
visited = set()
def build(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:   # 向输入方向走
            build(child)
        topo.append(v)          # 后序：最后加入自己
build(output)
# topo[-1] == output，topo[0] 是叶节点
```

### 箭头方向澄清：前向数据流 vs 拓扑依赖关系

这里很容易搞混。让我们明确两种"箭头"的方向：

**前向计算图的箭头**（数据如何流动）：
- 输入 → 运算符 → 输出
- 比如 `c = a + b` 的箭头指向 `c`（数据从 `a, b` 流向 `c`）

**拓扑排序里的依赖箭头**（谁依赖谁）：
- 这里说 `u → v` 表示 "`u` 依赖 `v`"——即 `v` 是 `u` 的输入
- 拓扑排序 要求：若 `u` 依赖 `v`，则 `u` 排在 `v` **后面**
- **这意味着拓扑排序里的箭头反向了！** 它指向输入方向

**换句话说**：反向传播时，我们沿着 `_prev` 指针向**输入方向**回溯，而不是沿前向箭头的方向。这是为什么要用 `for child in v._prev: build(child)`（逆向遍历），而不是 `for out in v._next:`（前向遍历）。

In [2]:
# 演示拓扑排序顺序
a = Value(2.0); b = Value(3.0); c = Value(-1.0)
L = a * b + c   # L = a*b + c

topo = []
visited = set()
def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)

build_topo(L)
print("拓扑序（从叶到输出）：")
for i, v in enumerate(topo):
    print(f"  [{i}] op={v._op!r:5s}  data={v.data:.2f}  prev_count={len(v._prev)}")
print()
print("逆序（backward 的遍历方向）：", [v._op or 'leaf' for v in reversed(topo)])

拓扑序（从叶到输出）：
  [0] op=''     data=-1.00  prev_count=0
  [1] op=''     data=2.00  prev_count=0
  [2] op=''     data=3.00  prev_count=0
  [3] op='*'    data=6.00  prev_count=2
  [4] op='+'    data=5.00  prev_count=2

逆序（backward 的遍历方向）： ['+', '*', 'leaf', 'leaf', 'leaf']


### 为什么后序遍历给出正确的拓扑序？（反例对比）

如果用**前序**加入（先 `topo.append(v)`，再递归），会发生什么？

```
def build_preorder(v):   # ❌ 前序（错误）
    if v not in visited:
        visited.add(v)
        topo.append(v)        # ← 先加入自己
        for child in v._prev:
            build_preorder(child)
```

这样得到的 `topo` 顺序会**反过来**——输出节点最先被加入，叶节点最后。当我们用 `reversed(topo)` 反转时，就变成了"输出节点最后"，这正好搞反了。

具体影响：逆序遍历时，会先对**输入节点**调用 `_backward()`，但它们的梯度还没从**输出**传来（因为输出还没处理），所以累积到的都是 0。

**证明：用前序遍历试试**

In [3]:
# 前序 vs 后序对比
a_demo = Value(2.0); b_demo = Value(3.0); c_demo = Value(-1.0)
L_demo = a_demo * b_demo + c_demo

# 后序遍历（正确）
topo_post = []
visited_post = set()
def build_postorder(v):
    if v not in visited_post:
        visited_post.add(v)
        for child in v._prev:
            build_postorder(child)
        topo_post.append(v)
build_postorder(L_demo)

# 前序遍历（错误）
topo_pre = []
visited_pre = set()
def build_preorder(v):
    if v not in visited_pre:
        visited_pre.add(v)
        topo_pre.append(v)        # ← 先加入
        for child in v._prev:
            build_preorder(child)
build_preorder(L_demo)

print("后序遍历（正确）：", [v._op or 'leaf' for v in topo_post])
print("  逆序后(backward遍历):", [v._op or 'leaf' for v in reversed(topo_post)])
print()
print("前序遍历（错误）：", [v._op or 'leaf' for v in topo_pre])
print("  逆序后(backward遍历):", [v._op or 'leaf' for v in reversed(topo_pre)])
print()
print("结论：前序的逆序是从叶向输出，会导致梯度计算顺序错误")

后序遍历（正确）： ['leaf', 'leaf', '*', 'leaf', '+']
  逆序后(backward遍历): ['+', 'leaf', '*', 'leaf', 'leaf']

前序遍历（错误）： ['+', '*', 'leaf', 'leaf', 'leaf']
  逆序后(backward遍历): ['leaf', 'leaf', 'leaf', '*', '+']

结论：前序的逆序是从叶向输出，会导致梯度计算顺序错误


### 递归过程详细演示：`L = a*b + c` 的完整 DFS 轨迹

让我们手工跟踪 `build_topo()` 对 `L = a*b + c` 的处理过程。计算图结构：

```
a ──┐
    ├─→ (a*b) ──┐
b ──┘           ├─→ L (output)
                │
c ──────────────┘
```

**递归调用顺序**（括号表示栈上的调用）：

| 步骤 | 递归栈 | 操作 | topo 当前内容 |
|------|--------|------|-------------|
| 1 | `build(L)` | L 未访问，标记访问 L | `[]` |
| 2 | `build(L) → build(mul_node)` | mul_node 未访问，标记访问 | `[]` |
| 3 | `build(mul) → build(a)` | a 未访问，标记访问，a 无子节点 | `[]` |
| 4 | `build(mul) ← a 返回` | a 没有更多子节点，**后序加入 a** | `[a]` |
| 5 | `build(mul) → build(b)` | b 未访问，标记访问，b 无子节点 | `[a]` |
| 6 | `build(mul) ← b 返回` | b 没有更多子节点，**后序加入 b** | `[a, b]` |
| 7 | `build(mul) ← 返回` | mul 已处理所有子节点，**后序加入 mul** | `[a, b, mul]` |
| 8 | `build(L) → build(c)` | c 未访问，标记访问，c 无子节点 | `[a, b, mul]` |
| 9 | `build(L) ← c 返回` | c 没有更多子节点，**后序加入 c** | `[a, b, mul, c]` |
| 10 | `build(L) ← 返回` | L 已处理所有子节点，**后序加入 L** | `[a, b, mul, c, L]` |

**最终结果**：`topo = [a, b, mul, c, L]`  
**逆序（backward 的遍历）**：`reversed(topo) = [L, c, mul, b, a]`

**为什么这个顺序是对的**：
- 先调 L 的 `_backward()`（初始化 L.grad = 1.0 后自动运行）
- 再调 c 的 `_backward()`（c 只出现在 L 的输入中，梯度来自 L）
- 再调 mul 的 `_backward()`（mul 也只出现在 L 的输入中）
- 最后才调 a、b 的 `_backward()`（它们出现在 mul 的输入中，梯度来自 mul）

这样每个节点被处理时，它的梯度已经从所有"下游使用它"的节点累积完毕。

## 概念 2：从输出节点出发，`self.grad = 1.0`

损失 `L` 对自身的梯度恒为 1：`dL/dL = 1`。
这是链式法则的起点——所有后续的 `_backward()` 都在把"上游梯度"（即 `out.grad`）乘以局部导数后写入输入节点的 `.grad`。

```
self.grad = 1.0          # dL/dL = 1
for node in reversed(topo):
    node._backward()     # 链式传播
```

如果忘记设置 `self.grad = 1.0`，所有乘以 `out.grad` 的项都变成 0，梯度全部为零。

In [4]:
# 演示：不设 grad=1.0 时梯度全零
a2 = Value(2.0); b2 = Value(3.0)
out2 = a2 * b2
# 手动调用 _backward，但不设置 out2.grad
out2._backward()
print("未设 out.grad=1.0 时：a2.grad =", a2.grad, "（预期 3.0，但得到 0）")

# 正确做法
a3 = Value(2.0); b3 = Value(3.0)
out3 = a3 * b3
out3.grad = 1.0          # ← 关键
out3._backward()
print("设置 out.grad=1.0 后：a3.grad =", a3.grad, "（预期 3.0）✅")

未设 out.grad=1.0 时：a2.grad = 0.0 （预期 3.0，但得到 0）
设置 out.grad=1.0 后：a3.grad = 3.0 （预期 3.0）✅


## 概念 3：梯度累积（`+=`）而非覆盖（`=`）

当同一个节点被多个后续节点使用时（菱形图），梯度来自多条路径，必须**相加**，不能用后一条路径覆盖前一条。

例：`L = a*a`（`a` 同时作为左右操作数）
- 路径1：`dL/d(左a) = a.data = 2`
- 路径2：`dL/d(右a) = a.data = 2`
- 总梯度：`a.grad = 2 + 2 = 4`（等于解析式 `dL/da = 2a = 4`）

在 `_backward` 里统一写 `self.grad += ...` 而非 `self.grad = ...`，自动处理菱形图。

In [5]:
# 菱形图演示：L = a * a
a4 = Value(2.0)
L4 = a4 * a4          # a4 同时是 _prev 里的两个"子节点"（同一对象）
L4.grad = 1.0
L4._backward()         # 执行一次 mul 的 _backward
print(f"a4.grad = {a4.grad:.1f}  （预期 4.0，因为 d(a^2)/da = 2a = 4）")
# 注意：mul 的 _backward 用 +=，所以两条路径都累积进来了

a4.grad = 4.0  （预期 4.0，因为 d(a^2)/da = 2a = 4）


### 深入理解：为什么 `+=` 累积正好对应多路径求导的相加？

这不是巧合，而是**链式法则本身的性质**。

**单条路径的链式法则**：
$$\frac{dL}{da} = \frac{dL}{d(\text{out})} \cdot \frac{d(\text{out})}{da}$$

**多条路径的情形**（如 `L = a*a`）：
- 路径1：`L → (第一个a)` 的贡献是 $\frac{dL}{d(\text{mul})} \cdot \frac{d(\text{mul})}{d(\text{left_a})} = 1 \cdot a = a$
- 路径2：`L → (第二个a)` 的贡献是 $\frac{dL}{d(\text{mul})} \cdot \frac{d(\text{mul})}{d(\text{right_a})} = 1 \cdot a = a$
- **总梯度**：两条路径相加：$a + a = 2a$

在代码中，这两条路径对应 `mul` 的 `_backward()` 中的两行赋值：
```python
def _bwd():
    self.grad  += other.data * out.grad    # ← 路径1：left_a 收到 other.data * out.grad
    other.grad += self.data  * out.grad    # ← 路径2：right_a 收到 self.data * out.grad
```

当 `self` 和 `other` 是同一个对象（即 `a`），这两行都对 `a.grad` 的同一个内存位置执行累加，就自动实现了多路径求导相加。

**这就是为什么必须用 `+=` 而不是 `=`**：`=` 会让后一条路径的贡献覆盖前一条，导致梯度计算错误。

## 1. ✏️ 实现 `Value.backward(self)`

**三步实现路线**：

| 步骤 | 动作 | 关键点 |
|---|---|---|
| 1 | DFS 后序遍历 `self` → `topo` 列表 | 用 `visited` set 避免重复；遍历 `node._prev` |
| 2 | `self.grad = 1.0` | dL/dL=1，是所有梯度计算的起点 |
| 3 | `for node in reversed(topo): node._backward()` | 逆序保证"上游先算完" |

**验证标准**：
```python
a, b, c = Value(2.0), Value(3.0), Value(-1.0)
L = a * b + c    # L.data = 5.0
L.backward()
assert abs(a.grad - 3.0) < 1e-12   # dL/da = b = 3
assert abs(b.grad - 2.0) < 1e-12   # dL/db = a = 2
assert abs(c.grad - 1.0) < 1e-12   # dL/dc = 1
```

**菱形图验证**（`L = a*a`，`a=2`）：
- `dL/da` 来自两条路径：左乘(`a.data=2`)＋右乘(`a.data=2`) → `grad = 4.0`
- 若误用 `=` 而非 `+=`，只保留最后一条路径，得到 `2.0`（错误）

In [6]:
def backward(self):
    # ✏️ TODO 步骤1：DFS 后序遍历，建立拓扑序列 topo
    topo = []
    visited = set()
    def build(v):
        raise NotImplementedError("TODO: 递归访问 v._prev，后序加入 topo")

    build(self)

    # ✏️ TODO 步骤2：设置输出节点梯度为 1.0
    # TODO: self.grad = 1.0

    # ✏️ TODO 步骤3：逆序调用每个节点的 _backward()
    pass

# 将方法绑定到 Value 类
Value.backward = backward  # 直接替换

In [7]:
# ⚠️ 请在完成上方练习格的 backward() 实现后再运行本格。未实现时会捕获提示而非崩溃。
try:
    # 检查：L = a*b + c
    a = Value(2.0); b = Value(3.0); c = Value(-1.0)
    L = a * b + c
    L.backward()
    
    assert abs(a.grad - 3.0) < 1e-9, f"a.grad 应为 3.0，得到 {a.grad}"
    assert abs(b.grad - 2.0) < 1e-9, f"b.grad 应为 2.0，得到 {b.grad}"
    assert abs(c.grad - 1.0) < 1e-9, f"c.grad 应为 1.0，得到 {c.grad}"
    print(f"a.grad={a.grad:.1f}  b.grad={b.grad:.1f}  c.grad={c.grad:.1f}")
    print("✅ backward() 基本检查通过")
    
    # 菱形图检查：L = a*a，dL/da = 2*a = 4
    a2 = Value(2.0)
    L2 = a2 * a2
    L2.backward()
    assert abs(a2.grad - 4.0) < 1e-9, f"a2.grad 应为 4.0，得到 {a2.grad}"
    print(f"菱形图 a2.grad={a2.grad:.1f}（预期 4.0）")
    print("✅ 菱形图梯度累积检查通过")
except (NotImplementedError, TypeError):
    print('⬜ TODO：请先在上方练习格中实现 backward()，再运行本格验证。')
except AssertionError as e:
    print(f'❌ 验证失败：{e}')
    print('💡 检查你的 topo 排序方向和 grad 累积逻辑。')


⬜ TODO：请先在上方练习格中实现 backward()，再运行本格验证。


## 参数实验：数值微分（numerical differentiation）验证梯度

用数值微分 `(f(x+h) - f(x-h)) / (2h)` 独立计算梯度，与 `backward()` 结果对比，差距应 `< 1e-4`。

实验参数：
- `h = 1e-5`（步长太大误差大，太小浮点抖动）
- 测试函数：`L = a*b + c.relu()`（含 relu 的混合表达式）
- 预期现象：`a.grad` 等于 `b.data`，数值微分与解析梯度差距 `< 1e-4`

## 如何验证梯度正确性？数值微分入门

### 为什么用数值微分？

`backward()` 实现完后，我们需要**独立地**检验梯度计算是否正确。数值微分（numerical differentiation）给了一个不依赖代码逻辑的参考值。

### 标准导数的定义和问题

高中数学的导数定义：
$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

但在计算机中，我们不能取极限。用有限的 $h$ 近似：
$$f'(x) \approx \frac{f(x+h) - f(x)}{h} \quad (\text{单边差分})$$

**问题**：这个近似误差很大。例如 $f(x) = x^2$ 在 $x=2$ 时，用 $h=0.001$ 得到的近似是 $\frac{(2.001)^2 - 2^2}{0.001} = 4.001$，而精确值是 $4.0$，差了 $0.001$。

### 中心差分：更精确的公式

用**两边对称**的点：
$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h} \quad (\text{中心差分})$$

**为什么更准**？用泰勒展开（不需要完全理解，只看结论）：
$$f(x+h) \approx f(x) + f'(x) \cdot h + \frac{f''(x)}{2} \cdot h^2 + O(h^3)$$
$$f(x-h) \approx f(x) - f'(x) \cdot h + \frac{f''(x)}{2} \cdot h^2 + O(h^3)$$

相减：$f(x+h) - f(x-h) \approx 2f'(x) \cdot h + O(h^3)$

所以中心差分的误差是 $O(h^2)$（二阶），而单边差分是 $O(h)$（一阶）。**中心差分的精度高一个数量级**。

### 步长 h 的选择：为什么是 1e-5？

- **太大** ($h = 0.1$)：近似误差大，数值梯度不准
- **太小** ($h = 1e-10$)：浮点数舍入误差主导，反而不准
- **最优范围**：通常 $h = 10^{-4}$ 到 $10^{-6}$ 之间

在这个范围内，泰勒展开的高阶项和浮点舍入误差都相对较小。$h = 10^{-5}$ 是一个经验上的折中值，许多深度学习框架都用这个。

**验证思路**：我们计算 `backward()` 的梯度和数值梯度，如果误差 `< 1e-4`，就说明 `backward()` 的实现是对的。

In [8]:
def numeric_grad(f, x, h=1e-5):
    """对标量输入 x 计算 f 的数值梯度（中心差分）。"""
    return (f(x + h) - f(x - h)) / (2 * h)

# 固定其他参数，逐一扰动输入
a_val, b_val, c_val = 2.0, 3.0, -0.5

def L_a(av): return (Value(av) * Value(b_val) + Value(c_val).relu()).data
def L_b(bv): return (Value(a_val) * Value(bv) + Value(c_val).relu()).data
def L_c(cv): return (Value(a_val) * Value(b_val) + Value(cv).relu()).data

num_a = numeric_grad(L_a, a_val)
num_b = numeric_grad(L_b, b_val)
num_c = numeric_grad(L_c, c_val)

# 解析梯度（backward）——依赖上方练习格的实现
try:
    a = Value(a_val); b = Value(b_val); c = Value(c_val)
    L = a * b + c.relu()
    L.backward()

    print(f"{'变量':<6} {'backward':>12} {'数值微分':>12} {'误差':>12}")
    for name, ana, num in [('a', a.grad, num_a), ('b', b.grad, num_b), ('c', c.grad, num_c)]:
        err = abs(ana - num)
        flag = "✅" if err < 1e-4 else "❌"
        print(f"{name:<6} {ana:>12.6f} {num:>12.6f} {err:>12.2e} {flag}")
except (NotImplementedError, TypeError):
    print(f"数值微分参考值：num_a={num_a:.6f}, num_b={num_b:.6f}, num_c={num_c:.6f}")
    print('⬜ TODO：实现 backward() 后重跑本格，对比解析梯度与数值微分。')

数值微分参考值：num_a=3.000000, num_b=2.000000, num_c=0.000000
⬜ TODO：实现 backward() 后重跑本格，对比解析梯度与数值微分。


## 🎯 未来的回报 (Future Payoff)

今天你亲手搞懂的**反向传播**，会在 **L57 / L58 / L60** 再次出现——L57 用它跑通整个 MLP 的反向，L58 靠它在训练循环里更新权重（`w.data -= lr * w.grad`），到 L60 你会发现 PyTorch 的 `loss.backward()` 做的正是你今天手写的这套图遍历。手写过一遍，之后调 autograd 时你永远知道引擎盖下发生了什么。

## 本课收束

`Value.backward()` 通过 DFS 后序遍历建立拓扑序，再逆序调用每个节点的 `_backward()`，输出每个叶节点的 `.grad`（即损失对该参数的偏导（partial derivative）数）。
梯度用 `+=` 累积，保证菱形图（参数复用）下多路径贡献正确相加。
此机制直接对应后续训练循环的参数更新步骤（见 **L58** `L58_training_loop.ipynb`）：`w.data -= lr * w.grad`——Aurora ML 引擎（`src/aurora/ml/`，计划中模块）的训练循环也遵循此模式。
下一节（**L57**）将在此基础上组装 `Neuron → Layer → MLP`，把多个 `Value` 节点搭成真实网络，跑通整网的前向 + 反向；参数更新的完整训练循环留到 L58。

## ✏️ 闭卷推导检查格 — 反向传播矩阵公式

**规则：关闭上方所有格，先在此 Markdown 格写出推导，再在下方 Code 格实现。**

**题目**：2 层 MLP 前向传播：
$$z^{(1)} = W^{(1)} x + b^{(1)}, \quad a^{(1)} = \sigma(z^{(1)})$$
$$\hat{y} = W^{(2)} a^{(1)} + b^{(2)}, \quad L = \tfrac{1}{2}(\hat{y} - y)^2$$

推导（写在此处，不查笔记）：

1. $\dfrac{\partial L}{\partial W^{(2)}} = ?$（写成矩阵外积形式）

2. $\delta^{(1)} = \dfrac{\partial L}{\partial z^{(1)}} = ?$（含 $\sigma'(z^{(1)})$ 的 Hadamard 积）

3. $\dfrac{\partial L}{\partial W^{(1)}} = ?$（写成矩阵外积形式）

（在此处写推导...）

## ReLU 梯度中的掩码操作详解

### ReLU 的数学定义和导数

ReLU 函数定义为：
$$\text{ReLU}(z) = \max(0, z) = \begin{cases} z & \text{if } z > 0 \\ 0 & \text{otherwise} \end{cases}$$

它的导数是分段函数：
$$\text{ReLU}'(z) = \frac{d}{dz} \text{ReLU}(z) = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{if } z \le 0 \end{cases}$$

**关键观察**：导数就像一个"开关"——只要激活被打开的位置（`z > 0`）的梯度才能通过，其他位置的梯度都被截断为 0。

### 为什么是掩码？

当 $z$ 是一个向量时，ReLU 的导数也是一个向量，每个位置对应一个 0 或 1：
$$\text{ReLU}'(z) = \begin{pmatrix} \mathbb{1}_{z_1 > 0} \\ \mathbb{1}_{z_2 > 0} \\ \vdots \\ \mathbb{1}_{z_n > 0} \end{pmatrix} = \begin{pmatrix} 1 \text{ or } 0 \\ 1 \text{ or } 0 \\ \vdots \end{pmatrix}$$

其中 $\mathbb{1}_{z_i > 0}$ 是**指示函数**：当条件为真时是 1，否则是 0。

这个 0/1 向量被称为**掩码（mask）**，因为在反向传播时，它"掩盖"了某些梯度分量。

### 在神经网络反向传播中的应用

考虑两层网络的反向传播：
$$\frac{\partial L}{\partial z^{(1)}} = \frac{\partial L}{\partial a^{(1)}} \odot \text{ReLU}'(z^{(1)})$$

其中 $\odot$ 是 Hadamard 积（逐元素相乘）。

**数值例子**：假设
$$z^{(1)} = \begin{pmatrix} 0.5 \\ -0.3 \\ 1.2 \\ -0.1 \end{pmatrix}, \quad \frac{\partial L}{\partial a^{(1)}} = \begin{pmatrix} 0.1 \\ 0.2 \\ 0.15 \\ 0.05 \end{pmatrix}$$

则掩码为：
$$\text{ReLU}'(z^{(1)}) = (z^{(1)} > 0) = \begin{pmatrix} 1 \\ 0 \\ 1 \\ 0 \end{pmatrix}$$

梯度为：
$$\frac{\partial L}{\partial z^{(1)}} = \begin{pmatrix} 0.1 \\ 0.2 \\ 0.15 \\ 0.05 \end{pmatrix} \odot \begin{pmatrix} 1 \\ 0 \\ 1 \\ 0 \end{pmatrix} = \begin{pmatrix} 0.1 \\ 0 \\ 0.15 \\ 0 \end{pmatrix}$$

**观察**：第 2、4 位置（对应 $z < 0$ 的地方）的梯度被**清零了**，因为 ReLU 在那些位置的导数是 0。

### 从掩码到权重梯度

紧接着，为了求 $\frac{\partial L}{\partial W^{(1)}}$，我们继续用链式法则：
$$\frac{\partial L}{\partial W^{(1)}} = \frac{\partial L}{\partial z^{(1)}} \otimes x^T = \begin{pmatrix} 0.1 \\ 0 \\ 0.15 \\ 0 \end{pmatrix} \otimes \begin{pmatrix} x_1 & x_2 & \cdots \end{pmatrix}$$

这样，ReLU 的掩码效果就**间接地**作用在权重梯度上：那些被 ReLU 掩掉的激活对应的权重行也不会被更新（梯度为 0）。

## 矩阵求导基础：外积与 Hadamard 积

到目前为止，我们处理的都是标量。但神经网络涉及矩阵和向量，梯度也是矩阵形式。本节补充两个关键概念。

### 外积（Outer Product）

**定义**：给定列向量 $u$ (形状 $m \times 1$) 和行向量 $v^T$ (形状 $1 \times n$)，外积定义为：
$$u \otimes v^T = u v^T = \begin{pmatrix} u_1 \\ u_2 \\ \vdots \\ u_m \end{pmatrix} \begin{pmatrix} v_1 & v_2 & \cdots & v_n \end{pmatrix} = \begin{pmatrix} u_1 v_1 & u_1 v_2 & \cdots & u_1 v_n \\ u_2 v_1 & u_2 v_2 & \cdots & u_2 v_n \\ \vdots & \vdots & \ddots & \vdots \\ u_m v_1 & u_m v_2 & \cdots & u_m v_n \end{pmatrix}$$

**结果**：一个 $m \times n$ 的矩阵，其中第 $(i,j)$ 元素是 $u_i \cdot v_j$。

**例子**：
$$u = \begin{pmatrix} 2 \\ 3 \end{pmatrix}, \quad v^T = \begin{pmatrix} 1 & 4 \end{pmatrix} \quad \Rightarrow \quad u v^T = \begin{pmatrix} 2 \cdot 1 & 2 \cdot 4 \\ 3 \cdot 1 & 3 \cdot 4 \end{pmatrix} = \begin{pmatrix} 2 & 8 \\ 3 & 12 \end{pmatrix}$$

### 为什么梯度是外积？维度角度

考虑矩阵乘法 $\hat{y} = W \cdot a$，其中：
- $W$ 是 $1 \times 4$ 的矩阵（权重，待求梯度）
- $a$ 是 $4 \times 1$ 的列向量（激活）
- $\hat{y}$ 是 $1 \times 1$ 的标量（输出）

损失 $L$ 对 $W$ 的梯度应该与 $W$ 形状相同（都是 $1 \times 4$）。

从链式法则：
$$\frac{\partial L}{\partial W_{ij}} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial W_{ij}}$$

注意到 $\hat{y} = \sum_k W_{ik} a_k$，所以 $\frac{\partial \hat{y}}{\partial W_{ij}} = a_j$。

因此：
$$\frac{\partial L}{\partial W_{ij}} = \frac{\partial L}{\partial \hat{y}} \cdot a_j$$

用矩阵形式写：
$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial \hat{y}} \otimes a^T$$

即"上游梯度"与"激活值转置"的外积。这保证了形状匹配：$(1 \times 1) \otimes (4 \times 1)^T = (1 \times 1) \otimes (1 \times 4) = (1 \times 4)$ ✓

### Hadamard 积（元素逐一相乘）

**定义**：两个形状相同的矩阵，对应位置元素相乘：
$$A \odot B = \begin{pmatrix} a_{11} b_{11} & a_{12} b_{12} & \cdots \\ a_{21} b_{21} & a_{22} b_{22} & \cdots \\ \vdots & \vdots & \ddots \end{pmatrix}$$

**符号**：在数学中记作 $A \odot B$，在 NumPy 中用 `*`（对数组）。

**例子**：
$$A = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}, \quad B = \begin{pmatrix} 5 & 6 \\ 7 & 8 \end{pmatrix} \quad \Rightarrow \quad A \odot B = \begin{pmatrix} 5 & 12 \\ 21 & 32 \end{pmatrix}$$

**与矩阵乘法 `@` 的区别**：
- Hadamard 积 `*`：逐元素相乘，结果形状与输入相同
- 矩阵乘法 `@`：行×列的线性组合，结果形状通常不同

### 在反向传播中的应用

当激活函数是非线性的（如 ReLU、Sigmoid），它的导数是**逐元素的**：
$$\text{ReLU}'(z) = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{otherwise} \end{cases}$$

写成向量形式，如果 $z$ 是一个向量，则：
$$\sigma'(z) = \begin{pmatrix} \sigma'(z_1) \\ \sigma'(z_2) \\ \vdots \end{pmatrix}$$

链式法则传播时，梯度与导数用 Hadamard 积相乘：
$$\frac{\partial L}{\partial z} = \frac{\partial L}{\partial a} \odot \sigma'(z)$$

这样每个位置的梯度都被对应位置的导数缩放（ReLU 为 0 或 1，Sigmoid 为 $\sigma(z)(1-\sigma(z))$ 等）。

In [9]:
# 外积与 Hadamard 积的具体演示
import numpy as np

print("=" * 60)
print("外积演示")
print("=" * 60)
u = np.array([[2], [3]])  # (2, 1) 列向量
v_T = np.array([[1, 4]])  # (1, 2) 行向量
outer_product = u @ v_T   # 外积
print(f"u = {u.T}  (shape {u.shape})")
print(f"v^T = {v_T}  (shape {v_T.shape})")
print(f"u ⊗ v^T = u @ v^T =\n{outer_product}  (shape {outer_product.shape})")
print()

print("=" * 60)
print("Hadamard 积演示")
print("=" * 60)
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
hadamard = A * B  # NumPy 的 * 对数组是 Hadamard 积
print(f"A =\n{A}")
print(f"B =\n{B}")
print(f"A ⊙ B (逐元素相乘) =\n{hadamard}")
print()

print("=" * 60)
print("实际例子：神经网络梯度的外积形式")
print("=" * 60)
# 假设：
# - W2 是 (1, 4) 权重矩阵
# - a1 是 (4, 1) 激活向量  
# - dL/dyhat 是 (1, 1) 标量梯度（或看作 1 维）
W2 = np.array([[0.1, 0.2, 0.3, 0.4]])  # (1, 4)
a1 = np.array([[1], [2], [3], [4]])    # (4, 1)
dL_dyhat = np.array([[0.5]])           # (1, 1) 标量损失对输出的梯度

# 梯度用外积计算
dL_dW2 = dL_dyhat @ a1.T  # (1, 1) @ (4, 1) -> (1, 4) ✓
print(f"dL/dyhat = {dL_dyhat.T}  (shape {dL_dyhat.shape})")
print(f"a1 = {a1.T}  (shape {a1.shape})")
print(f"dL/dW2 = dL/dyhat ⊗ a1^T =")
print(f"{dL_dW2}  (shape {dL_dW2.shape})")
print()

print("验证：dL/dW2 的每个元素")
for i in range(W2.shape[0]):
    for j in range(W2.shape[1]):
        elem = dL_dyhat[i, 0] * a1[j, 0]
        print(f"  dL/dW2[{i},{j}] = dL/dyhat × a1[{j}] = {dL_dyhat[i,0]} × {a1[j,0]} = {elem}")
print()

print("=" * 60)
print("ReLU 中的 Hadamard 积示例")
print("=" * 60)
z1 = np.array([[0.5], [-0.3], [1.2], [-0.1]])  # (4, 1)
mask = (z1 > 0).astype(float)  # ReLU 导数（作为掩码）
dL_da1 = np.array([[0.1], [0.2], [0.15], [0.05]])  # (4, 1)
dL_dz1 = dL_da1 * mask  # Hadamard 积
print(f"z1 = {z1.T}")
print(f"mask (z1 > 0) = {mask.T}")
print(f"dL/da1 = {dL_da1.T}")
print(f"dL/dz1 = dL/da1 ⊙ mask(z1) = {dL_dz1.T}")
print("(注意：负数位置的梯度被掩码清零了，因为 ReLU 在那里的导数是 0)")

外积演示
u = [[2 3]]  (shape (2, 1))
v^T = [[1 4]]  (shape (1, 2))
u ⊗ v^T = u @ v^T =
[[ 2  8]
 [ 3 12]]  (shape (2, 2))

Hadamard 积演示
A =
[[1 2]
 [3 4]]
B =
[[5 6]
 [7 8]]
A ⊙ B (逐元素相乘) =
[[ 5 12]
 [21 32]]

实际例子：神经网络梯度的外积形式
dL/dyhat = [[0.5]]  (shape (1, 1))
a1 = [[1 2 3 4]]  (shape (4, 1))
dL/dW2 = dL/dyhat ⊗ a1^T =
[[0.5 1.  1.5 2. ]]  (shape (1, 4))

验证：dL/dW2 的每个元素
  dL/dW2[0,0] = dL/dyhat × a1[0] = 0.5 × 1 = 0.5
  dL/dW2[0,1] = dL/dyhat × a1[1] = 0.5 × 2 = 1.0
  dL/dW2[0,2] = dL/dyhat × a1[2] = 0.5 × 3 = 1.5
  dL/dW2[0,3] = dL/dyhat × a1[3] = 0.5 × 4 = 2.0

ReLU 中的 Hadamard 积示例
z1 = [[ 0.5 -0.3  1.2 -0.1]]
mask (z1 > 0) = [[1. 0. 1. 0.]]
dL/da1 = [[0.1  0.2  0.15 0.05]]
dL/dz1 = dL/da1 ⊙ mask(z1) = [[0.1  0.   0.15 0.  ]]
(注意：负数位置的梯度被掩码清零了，因为 ReLU 在那里的导数是 0)


In [10]:
# 验证：在下方 Code 格实现你推导的公式，与数值梯度对比
# 只需填写标注 TODO 的三行；其余代码已给出
import numpy as np

def sigmoid(z): return 1 / (1 + np.exp(-z))
def dsigmoid(z): s = sigmoid(z); return s * (1 - s)

np.random.seed(7)
W1 = np.random.randn(4, 3) * 0.1
b1 = np.zeros((4, 1))
W2 = np.random.randn(1, 4) * 0.1
b2 = np.zeros((1, 1))
x  = np.random.randn(3, 1)
y  = np.array([[1.0]])

# 前向传播
z1 = W1 @ x + b1; a1 = sigmoid(z1)
z2 = W2 @ a1 + b2; y_hat = z2
loss = 0.5 * (y_hat - y) ** 2

# ── 反向传播（按你的推导替换三个 zeros 占位） ────────────────────
dL_dyhat = y_hat - y
# ✏️ TODO 1: dL/dW2  （shape 与 W2 相同，dL_dyhat 和 a1 的外积）
dL_dW2 = np.zeros_like(W2)        # ← 替换为你的推导

# ✏️ TODO 2: δ¹ = dL/dz¹  （shape 与 z1 相同，W2 转置回传后与 σ'(z1) 做 Hadamard 积）
delta1 = np.zeros_like(z1)        # ← 替换为你的推导

# ✏️ TODO 3: dL/dW1  （shape 与 W1 相同，delta1 和 x 的外积）
dL_dW1 = np.zeros_like(W1)        # ← 替换为你的推导

# 数值梯度验证（有限差分）——参考值，不可改动
def loss_fn(W2_, W1_):
    a1_ = sigmoid(W1_ @ x + b1)
    yh  = W2_ @ a1_ + b2
    return 0.5 * ((yh - y) ** 2).item()   # .item()：(1,1) 数组→标量（numpy 2.x 下 float() 会报错）

eps = 1e-5
def num_grad(param, i, j, loss_fn_partial):
    p = param.copy(); p[i,j] += eps
    return (loss_fn_partial(p) - loss_fn_partial(param.copy())) / eps

if not (dL_dW2.any() or delta1.any() or dL_dW1.any()):
    print('⬜ TODO：先把三个 zeros 占位替换为你的手推公式，再运行本格做数值验证。')
else:
    ng_W2_00 = num_grad(W2, 0, 0, lambda w: loss_fn(w, W1))
    ng_W1_00 = num_grad(W1, 0, 0, lambda w: loss_fn(W2, w))

    tol = 1e-4
    ok_W2 = abs(dL_dW2[0,0] - ng_W2_00) < tol
    ok_W1 = abs(dL_dW1[0,0] - ng_W1_00) < tol

    print(f"dL/dW2[0,0]: 你的推导 = {dL_dW2[0,0]:+.6f}, 数值梯度 = {ng_W2_00:+.6f}  {'✅' if ok_W2 else '❌'}")
    print(f"dL/dW1[0,0]: 你的推导 = {dL_dW1[0,0]:+.6f}, 数值梯度 = {ng_W1_00:+.6f}  {'✅' if ok_W1 else '❌'}")
    if ok_W2 and ok_W1:
        print("\n✅ 反向传播推导验证通过")
    else:
        print("\n❌ 公式有误，请对照推导重新检查 TODO 行")

⬜ TODO：先把三个 zeros 占位替换为你的手推公式，再运行本格做数值验证。


## ✏️ 推导练习：手推两层网络的梯度

给定两层网络：`loss = ((W2 @ relu(W1 @ x)) - y)²`

**闭卷任务**：不看上方代码，在下方推导：
1. `∂loss/∂W2` = ？（提示：令 `h = relu(W1 @ x)`，`ŷ = W2 @ h`）
2. `∂loss/∂W1` = ？（提示：链式法则经过 relu）

在下方代码格中：用数值微分（有限差分法）验证你的推导是否正确。

In [11]:
import numpy as np

# 简单两层网络（标量输出）
np.random.seed(0)
x  = np.array([1.0, 2.0])
y  = np.array([1.0])
W1 = np.random.randn(3, 2) * 0.5
W2 = np.random.randn(1, 3) * 0.5

def forward(W1, W2, x, y):
    h    = np.maximum(0, W1 @ x)   # relu
    yhat = W2 @ h
    loss = float(((yhat - y) ** 2).sum())
    return loss, h, yhat

loss, h, yhat = forward(W1, W2, x, y)

# ✏️ TODO: 把 ... 替换为你的手推公式（提示：∂loss/∂ŷ = 2(ŷ-y)）
dL_dW2 = ...   # shape 与 W2 相同（提示：某个向量与 h 的外积）
dL_dW1 = ...   # shape 与 W1 相同（提示：回传过 relu 掩码 (W1@x > 0)，再与 x 外积）

# --- 数值梯度验证（中心差分，填完手推后自动运行） ---
if dL_dW2 is ... or dL_dW1 is ...:
    print('⬜ TODO：先填入 dL_dW2 / dL_dW1 的手推公式，再运行本格做数值验证。')
else:
    eps = 1e-5
    num_grad_W2 = np.zeros_like(W2)
    for i in range(W2.shape[0]):
        for j in range(W2.shape[1]):
            W2p = W2.copy(); W2p[i,j] += eps
            W2m = W2.copy(); W2m[i,j] -= eps
            num_grad_W2[i,j] = (forward(W1,W2p,x,y)[0] - forward(W1,W2m,x,y)[0]) / (2*eps)
    np.testing.assert_allclose(dL_dW2, num_grad_W2, atol=1e-5)
    print("✅ dL/dW2 数值验证通过")

    num_grad_W1 = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            W1p = W1.copy(); W1p[i,j] += eps
            W1m = W1.copy(); W1m[i,j] -= eps
            num_grad_W1[i,j] = (forward(W1p,W2,x,y)[0] - forward(W1m,W2,x,y)[0]) / (2*eps)
    np.testing.assert_allclose(dL_dW1, num_grad_W1, atol=1e-5)
    print("✅ dL/dW1 数值验证通过")

⬜ TODO：先填入 dL_dW2 / dL_dW1 的手推公式，再运行本格做数值验证。


In [ ]:
# ✏️ 本课自评
l56_review = {
    "topo_order_understood":   None,  # 理解为什么必须逆拓扑序才能正确反向传播？True/False
    "backward_implemented":    None,  # backward() DFS+逆序+grad累积 全部正确实现？True/False
    "grad_init_understood":    None,  # 记住 L.grad=1.0 是链式法则起点，不是任意选的？True/False
    "accumulation_vs_assign":  None,  # 菱形图演示中 += 与 = 的区别验证通过？True/False
    "whiteboard_passed":       None,  # 白板手推反向传播矩阵公式推导通过？True/False
}

unfilled = [k for k, v in l56_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l56_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L56 全部通关！进入 L57：MLP 从零搭建')

---

→ **下一课**　[L57 · MLP 从零搭建](L57_mlp.ipynb)

> 下节课将学习 **MLP 从零搭建**：手写全连接层、激活函数、前向 / 反向完整实现。